# 014 - Mistura XGBoost 007 + MLP ponderado 012

## Objetivo e protocolo

Comparamos o XGBoost 007 (200 árvores, `max_depth=5`, `learning_rate=0.1`, `multi:softprob`, `hist`, `random_state=42`, `n_jobs=1`) com o MLP ponderado 012 (64 → 32, ReLU, 16 logits, Adam `lr=0.001`, `weight_decay=0`, 12 épocas). Reutilizamos exatamente a divisão e validação interna do 012. O XGBoost usa preparação sem escala; o MLP usa escala numérica. Durante seleção cada pré-processador e os pesos MLP são ajustados só no treino interno.

Comparamos os modelos individuais e misturas XGBoost/MLP 90/10, 75/25 e 50/50 por Macro F1 interna. Selecionamos a melhor mistura sem consultar a validação fixa. Depois refazemos ambos nos 80%, avaliamos uma vez nos 20%, e finalmente treinamos nos dados rotulados completos para exportar. A pontuação Kaggle será registada após submissão.

## Dados e partições idênticas ao 012

A leitura, preparação e criação das duas partições vêm do notebook 012. Confirmamos os índices de X/y e que as divisões não se sobrepõem. A validação fixa fica reservada até à avaliação dos modelos finais.

In [1]:
import sys
import matplotlib.pyplot as plt
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
import torch
from sklearn.model_selection import train_test_split

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')
sys.path.insert(0, str(repo_root))
from src.preparar_dados import build_preprocessor, make_model_pipeline, make_stratified_split, preparar_dados

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f'CSV de treino/teste não encontrados em {repo_root / "notebooks"}')

print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}; scikit-learn: {sklearn.__version__}')
print(f'Dispositivo: CPU')
print(f'Raiz do projeto: {repo_root}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X_train, X_val, y_train, y_val = make_stratified_split(
    prepared['X_train'], prepared['y_train'], test_size=0.2, random_state=42
)
assert len(X_train) == 8769 and len(X_val) == 2193
assert sorted(y_train.unique().tolist()) == list(range(16))
print(f'Treino da experiência: {X_train.shape}; validação fixa reservada: {X_val.shape}')

X_inner_train, X_inner_val, y_inner_train, y_inner_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train,
)
assert X_inner_train.columns.equals(X_inner_val.columns)
print(f'Treino interno: {X_inner_train.shape}; validação interna: {X_inner_val.shape}')
assert X_train.index.equals(y_train.index) and X_val.index.equals(y_val.index)
assert X_inner_train.index.equals(y_inner_train.index) and X_inner_val.index.equals(y_inner_val.index)
assert X_train.index.intersection(X_val.index).empty
assert X_inner_train.index.intersection(X_inner_val.index).empty
print('X/y índices alinhados; divisões interna e fixa disjuntas.')


Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
PyTorch: 2.15.0; scikit-learn: 1.6.1
Dispositivo: CPU
Raiz do projeto: /Users/eduardaferreira/daa-kaggle-2026-2027


Treino da experiência: (8769, 236); validação fixa reservada: (2193, 236)
Treino interno: (7453, 236); validação interna: (1316, 236)
X/y índices alinhados; divisões interna e fixa disjuntas.


## Configuração XGBoost e helpers do MLP

O XGBoost reproduz a configuração vencedora do 007 sem escala numérica. O MLP reutiliza a arquitetura, a fórmula de pesos inversos, o pré-processamento normalizado e o treino determinístico do 012. As matrizes de probabilidade são alinhadas explicitamente às classes 0–15.

In [2]:
import random
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import xgboost
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

SEED = 42
BATCH_SIZE = 128
MLP_EPOCHS = 12
LEARNING_RATE = 0.001
NUM_CLASSES = 16
CLASS_IDS = np.arange(NUM_CLASSES)
MIXTURE_MLP_WEIGHTS = [0.10, 0.25, 0.50]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

def to_dense_float32(matrix):
    if hasattr(matrix, 'toarray'):
        matrix = matrix.toarray()
    return np.asarray(matrix, dtype=np.float32)

def inverse_frequency_weights(targets):
    counts = np.bincount(np.asarray(targets, dtype=np.int64), minlength=NUM_CLASSES)
    if np.any(counts == 0):
        raise ValueError(f'Classes ausentes ao calcular pesos: {np.flatnonzero(counts == 0).tolist()}')
    return torch.tensor(len(targets) / (NUM_CLASSES * counts), dtype=torch.float32)

class AgeRangeMLP(torch.nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(input_features, 64), torch.nn.ReLU(),
            torch.nn.Linear(64, 32), torch.nn.ReLU(),
            torch.nn.Linear(32, NUM_CLASSES),
        )
    def forward(self, features):
        return self.layers(features)

def make_mlp(input_features, seed=SEED):
    torch.manual_seed(seed)
    return AgeRangeMLP(input_features).to('cpu')

def make_xgb():
    return make_model_pipeline(
        XGBClassifier(
            objective='multi:softprob', num_class=NUM_CLASSES, tree_method='hist',
            n_estimators=200, max_depth=5, learning_rate=0.1,
            random_state=SEED, n_jobs=1, verbosity=0,
        ),
        normalize_numeric=False,
    )

def align_probabilities(probabilities, model_classes):
    aligned = np.zeros((probabilities.shape[0], NUM_CLASSES), dtype=np.float64)
    for column, class_id in enumerate(model_classes):
        class_id = int(class_id)
        if class_id not in CLASS_IDS:
            raise ValueError(f'Classe inesperada no modelo: {class_id}')
        aligned[:, class_id] = probabilities[:, column]
    if not np.allclose(aligned.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError('As probabilidades alinhadas não somam 1.')
    return aligned

def train_mlp(features, targets, epochs):
    weights = inverse_frequency_weights(targets)
    criterion = torch.nn.CrossEntropyLoss(weight=weights)
    model = make_mlp(features.shape[1], seed=SEED)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=0)
    dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(np.asarray(targets, dtype=np.int64)))
    generator = torch.Generator(device='cpu').manual_seed(SEED)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator, num_workers=0)
    for _ in range(epochs):
        model.train()
        for batch_features, batch_targets in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_features), batch_targets)
            loss.backward()
            optimizer.step()
    return model, weights

def mlp_probabilities(model, features):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.from_numpy(features)), dim=1).cpu().numpy()

def macro_f1_from_proba(targets, probabilities):
    return f1_score(targets, probabilities.argmax(axis=1), labels=CLASS_IDS, average='macro', zero_division=0)

print(f'XGBoost {xgboost.__version__}; MLP epochs={MLP_EPOCHS}; classes={CLASS_IDS.tolist()}')

XGBoost 3.4.1; MLP epochs=12; classes=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


## Seleção na validação interna

Ajustamos o pipeline XGBoost e o pré-processador do MLP separadamente no treino interno. Os pesos MLP vêm apenas de `y_inner_train`. A validação interna fornece as probabilidades, Macro F1 individual e de cada mistura. Selecionamos a maior Macro F1 entre as três misturas definidas.

In [3]:
# XGBoost: pipeline comum, preparação ajustada só em X_inner_train.
xgb_inner = make_xgb()
xgb_inner.fit(X_inner_train, y_inner_train)
xgb_model_inner = xgb_inner.named_steps['model']
xgb_inner_val_proba = align_probabilities(
    xgb_inner.predict_proba(X_inner_val), xgb_model_inner.classes_
)

# MLP: scaler ajustado só no treino interno; pesos calculados só nos respetivos alvos.
mlp_inner_preprocessor = build_preprocessor(X_inner_train, normalize_numeric=True)
X_inner_train_mlp = to_dense_float32(mlp_inner_preprocessor.transform(X_inner_train))
X_inner_val_mlp = to_dense_float32(mlp_inner_preprocessor.transform(X_inner_val))
y_inner_train_array = y_inner_train.to_numpy(dtype=np.int64)
y_inner_val_array = y_inner_val.to_numpy(dtype=np.int64)
mlp_inner, mlp_inner_weights = train_mlp(X_inner_train_mlp, y_inner_train_array, MLP_EPOCHS)
mlp_inner_val_proba = mlp_probabilities(mlp_inner, X_inner_val_mlp)

internal_candidates = {
    'XGBoost individual': xgb_inner_val_proba,
    'MLP individual': mlp_inner_val_proba,
}
for mlp_weight in MIXTURE_MLP_WEIGHTS:
    internal_candidates[f'Mistura XGB/MLP {1-mlp_weight:.0%}/{mlp_weight:.0%}'] = (
        (1 - mlp_weight) * xgb_inner_val_proba + mlp_weight * mlp_inner_val_proba
    )
internal_scores = pd.DataFrame([
    {'configuracao': name, 'macro_f1_interna': macro_f1_from_proba(y_inner_val_array, probabilities)}
    for name, probabilities in internal_candidates.items()
])
display(internal_scores.sort_values('macro_f1_interna', ascending=False).reset_index(drop=True))
mixture_scores = internal_scores[internal_scores['configuracao'].str.startswith('Mistura')].sort_values(
    'macro_f1_interna', ascending=False, kind='stable'
).reset_index(drop=True)
selected_mixture = str(mixture_scores.iloc[0]['configuracao'])
selected_mlp_weight = next(w for w in MIXTURE_MLP_WEIGHTS if selected_mixture.endswith(f'{w:.0%}'))
selected_internal_macro_f1 = float(mixture_scores.iloc[0]['macro_f1_interna'])
xgb_internal_macro_f1 = float(internal_scores.loc[internal_scores['configuracao'].eq('XGBoost individual'), 'macro_f1_interna'].iloc[0])
mixture_beats_xgb = selected_internal_macro_f1 > xgb_internal_macro_f1
submission_is_exploratory = not mixture_beats_xgb
print(f'Mistura selecionada: {selected_mixture}; Macro F1 interna={selected_internal_macro_f1:.4f}')
print(f'XGBoost sozinho: Macro F1 interna={xgb_internal_macro_f1:.4f}')
print('A mistura superou XGBoost interno.' if mixture_beats_xgb else 'A mistura não superou XGBoost: submissão exploratória.')

,configuracao,macro_f1_interna
0,Mistura XGB/MLP 75%/25%,0.481544
1,Mistura XGB/MLP 50%/50%,0.478026
2,Mistura XGB/MLP 90%/10%,0.474492
3,MLP individual,0.474236
4,XGBoost individual,0.467955


Mistura selecionada: Mistura XGB/MLP 75%/25%; Macro F1 interna=0.4815
XGBoost sozinho: Macro F1 interna=0.4680
A mistura superou XGBoost interno.


## Reajuste nos 80% e avaliação única nos 20% fixos

Ajustamos dois novos modelos nos 80%: pipeline XGBoost sem escala e MLP com pré-processador normalizado e pesos recalculados nos 80%. Avaliamos XGBoost, MLP e mistura selecionada uma vez nos mesmos 20% fixos. A mistura é exploratória se ficar abaixo do XGBoost individual. Destacamos Macro F1, accuracy e F1/recall por classe, incluindo classes 1, 6 e 13.

In [4]:
# XGBoost e MLP novos ajustados nos 80% completos.
xgb_train = make_xgb()
xgb_train.fit(X_train, y_train)
xgb_model = xgb_train.named_steps['model']
xgb_val_proba = align_probabilities(xgb_train.predict_proba(X_val), xgb_model.classes_)

mlp_train_preprocessor = build_preprocessor(X_train, normalize_numeric=True)
X_train_mlp = to_dense_float32(mlp_train_preprocessor.transform(X_train))
X_val_mlp = to_dense_float32(mlp_train_preprocessor.transform(X_val))
y_train_array = y_train.to_numpy(dtype=np.int64)
y_val_array = y_val.to_numpy(dtype=np.int64)
mlp_train, weights_80 = train_mlp(X_train_mlp, y_train_array, MLP_EPOCHS)
mlp_val_proba = mlp_probabilities(mlp_train, X_val_mlp)

selected_proba = (1 - selected_mlp_weight) * xgb_val_proba + selected_mlp_weight * mlp_val_proba
validation_probas = {
    'XGBoost 007': xgb_val_proba,
    'MLP 012': mlp_val_proba,
    f'014 {selected_mixture}': selected_proba,
}
validation_rows = []
validation_reports = {}
for name, probabilities in validation_probas.items():
    predictions = probabilities.argmax(axis=1)
    report = classification_report(
        y_val_array, predictions, labels=CLASS_IDS, target_names=[str(i) for i in CLASS_IDS],
        output_dict=True, zero_division=0,
    )
    validation_reports[name] = report
    validation_rows.append({
        'modelo': name,
        'macro_f1': f1_score(y_val_array, predictions, labels=CLASS_IDS, average='macro', zero_division=0),
        'accuracy': accuracy_score(y_val_array, predictions),
    })
validation_summary = pd.DataFrame(validation_rows)
display(validation_summary)

selected_report = validation_reports[f'014 {selected_mixture}']
per_class = pd.DataFrame([
    {'classe': class_id, 'F1': selected_report[str(class_id)]['f1-score'],
     'recall': selected_report[str(class_id)]['recall'], 'support': int(selected_report[str(class_id)]['support'])}
    for class_id in CLASS_IDS
])
display(per_class)
key_class_rows = per_class[per_class['classe'].isin([1, 6, 13])].copy()
key_class_rows['escalão'] = key_class_rows['classe'].map({1:'18–19 anos',6:'40–44 anos',13:'75–79 anos'})
key_class_rows['F1_XGBoost_007'] = [0.31, 0.37, 0.28]
key_class_rows['recall_XGBoost_007'] = [0.24, 0.38, 0.27]
key_class_rows['F1_MLP_012'] = [0.471698, 0.131148, 0.125604]
key_class_rows['recall_MLP_012'] = [0.735294, 0.093023, 0.078788]
display(key_class_rows)
print('Resultados de referência fixa: XGBoost 007 Macro F1=0.4491 (Kaggle 0.45); MLP 012 Macro F1=0.4073, accuracy=0.4168, recall classe 1=25/34.')
print(f'014: mistura vs XGBoost Macro F1: {validation_rows[2]["macro_f1"] - validation_rows[0]["macro_f1"]:+.4f}; exploratória={submission_is_exploratory}')

,modelo,macro_f1,accuracy
0,XGBoost 007,0.449061,0.458276
1,MLP 012,0.407280,0.416781
2,014 Mistura XGB/MLP 75%/25%,0.461326,0.463748


,classe,F1,recall,support
0,0,0.813953,0.853659,41
1,1,0.448276,0.382353,34
2,2,0.583333,0.612500,80
3,3,0.340426,0.320000,75
4,4,0.274510,0.233333,90
5,5,0.362369,0.403101,129
6,6,0.358025,0.337209,172
7,7,0.405714,0.410405,173
8,8,0.600522,0.635359,181
9,9,0.616915,0.610837,203


,classe,F1,recall,support,escalão,F1_XGBoost_007,recall_XGBoost_007,F1_MLP_012,recall_MLP_012
1,1,0.448276,0.382353,34,18–19 anos,0.31,0.24,0.471698,0.735294
6,6,0.358025,0.337209,172,40–44 anos,0.37,0.38,0.131148,0.093023
13,13,0.237288,0.212121,165,75–79 anos,0.28,0.27,0.125604,0.078788


Resultados de referência fixa: XGBoost 007 Macro F1=0.4491 (Kaggle 0.45); MLP 012 Macro F1=0.4073, accuracy=0.4168, recall classe 1=25/34.
014: mistura vs XGBoost Macro F1: +0.0123; exploratória=False


## Treino completo, exportação e comparação com submissões anteriores

Treinamos novos modelos em todas as 10.962 linhas rotuladas. O MLP recalcula os pesos em todo o conjunto; cada modelo mantém a preparação própria. Geramos previsões na mesma ordem do teste e copiamos `RowId` do exemplo oficial. O CSV é criado sem índice e não substitui ficheiros existentes. A célula compara previsões com os CSVs 012, 013 e 007 disponíveis; a comparação com 015 é apresentada no notebook 015 depois da sua geração.

In [5]:
from pathlib import Path

submission_path = repo_root / 'submissions' / 'submission_014_xgboost_mlp.csv'
if submission_path.exists():
    raise FileExistsError(f'Não vou substituir submissão existente: {submission_path}')
example_candidates = [repo_root/'example_submission_10.csv', repo_root/'notebooks'/'example_submission_10.csv', Path.home()/'.Trash'/'example_submission_10 21.31.28.csv']
example_path = next((p for p in example_candidates if p.is_file()), None)
if example_path is None:
    raise FileNotFoundError('Exemplo oficial com RowId não encontrado.')
example = pd.read_csv(example_path)
if not {'RowId','AGE_RANGE'}.issubset(example.columns):
    raise ValueError(f'Formato inesperado do exemplo: {example.columns.tolist()}')
row_reference = example[['RowId']].copy()
if row_reference.RowId.isna().any() or row_reference.RowId.duplicated().any():
    raise ValueError('RowId oficial contém valores ausentes ou duplicados.')
X_all = prepared['X_train']; X_test = prepared['X_test']; y_all = prepared['y_train']
if len(X_all) != 10962 or len(X_test) != 3655 or len(row_reference) != 3655:
    raise ValueError('Dimensões inesperadas para treino/teste/RowId.')

xgb_all = make_xgb().fit(X_all, y_all)
xgb_all_model = xgb_all.named_steps['model']
xgb_test_proba = align_probabilities(xgb_all.predict_proba(X_test), xgb_all_model.classes_)
mlp_all_preprocessor = build_preprocessor(X_all, normalize_numeric=True)
X_all_mlp = to_dense_float32(mlp_all_preprocessor.transform(X_all))
X_test_mlp = to_dense_float32(mlp_all_preprocessor.transform(X_test))
y_all_array = y_all.to_numpy(dtype=np.int64)
mlp_all, weights_all = train_mlp(X_all_mlp, y_all_array, MLP_EPOCHS)
mlp_test_proba = mlp_probabilities(mlp_all, X_test_mlp)
test_proba = (1-selected_mlp_weight)*xgb_test_proba + selected_mlp_weight*mlp_test_proba
test_predictions_014 = test_proba.argmax(axis=1).astype(np.int64)
if len(test_predictions_014) != len(row_reference): raise ValueError('Comprimento das previsões difere dos RowId.')
submission_014 = row_reference.copy(); submission_014['AGE_RANGE'] = test_predictions_014
if submission_014.columns.tolist() != ['RowId','AGE_RANGE'] or not pd.api.types.is_integer_dtype(submission_014.AGE_RANGE) or not submission_014.AGE_RANGE.between(0,15).all():
    raise ValueError('Formato ou classes inválidas antes de guardar.')
submission_path.parent.mkdir(parents=True,exist_ok=True)
submission_014.to_csv(submission_path,index=False)
check_014=pd.read_csv(submission_path)
if check_014.shape != (3655,2) or check_014.columns.tolist()!=['RowId','AGE_RANGE'] or not check_014.RowId.equals(row_reference.RowId):
    raise ValueError('Validação de forma/ordem do CSV 014 falhou.')
if not pd.api.types.is_integer_dtype(check_014.AGE_RANGE) or not check_014.AGE_RANGE.between(0,15).all():
    raise ValueError('Classes no CSV 014 não são inteiras em [0,15].')
print(f'Guardado: {submission_path}; mistura={selected_mixture}; exploratória={submission_is_exploratory}')
print('CSV validado:',check_014.shape,check_014.columns.tolist(),'sem índice adicional')

previous_paths = {
    '012': repo_root/'submissions'/'submission_012_mlp_perda_ponderada.csv',
    '013': repo_root/'submissions'/'submission_013_mlp_arquitetura.csv',
    '007': repo_root/'submissions'/'submission_007_xgboost.csv',
}
comparison_rows=[]
for name, previous_path in previous_paths.items():
    if previous_path.is_file():
        previous=pd.read_csv(previous_path)
        if previous.columns.tolist()!=['RowId','AGE_RANGE'] or not previous.RowId.equals(check_014.RowId):
            raise ValueError(f'Formato/ordem incompatível em {previous_path}')
        comparison_rows.append({'comparado_com':name,'linhas_diferentes':int((check_014.AGE_RANGE != previous.AGE_RANGE).sum()),'total':len(check_014)})
display(pd.DataFrame(comparison_rows))
print('014 difere de 012:',int((check_014.AGE_RANGE != pd.read_csv(previous_paths['012']).AGE_RANGE).sum()) if previous_paths['012'].is_file() else 'CSV ausente')

Guardado: /Users/eduardaferreira/daa-kaggle-2026-2027/submissions/submission_014_xgboost_mlp.csv; mistura=Mistura XGB/MLP 75%/25%; exploratória=False
CSV validado: (3655, 2) ['RowId', 'AGE_RANGE'] sem índice adicional


,comparado_com,linhas_diferentes,total
0,012,1213,3655
1,007,464,3655


014 difere de 012: 1213


## Conclusão e Decisão
- Data: 10/11
- Pontuação Kaggle: 0.47
- Conclusão: A mistura de 75% XGBoost e 25% MLP obteve o melhor resultado até agora. Superou o XGBoost 007 tanto na validação fixa (0,4613 vs. 0,4491) como no Kaggle (0,47 vs. 0,45). Os 75–79 anos continuam a ser um dos escalões mais difíceis (F1 de 0,2373).
- Melhorias: Testar a combinação com o MLP 015, mantendo inicialmente os pesos 75/25, para medir o efeito de trocar apenas o MLP.

- A mistura **XGBoost/MLP 75%/25%** foi a melhor na validação interna (Macro F1 **0,4815**) e superou o XGBoost interno (0,4680). Na validação fixa, a mistura alcançou Macro F1 **0,4613** e accuracy **0,4637**, acima do XGBoost 007 (**0,4491**, Kaggle **0,45**) e do MLP 012 (**0,4073**, accuracy **0,4168**); não é exploratória. Nos escalões em destaque, mistura: classe 1 F1 **0,4483**, recall **0,3824**; classe 6 F1 **0,3580**, recall **0,3372**; classe 13 F1 **0,2373**, recall **0,2121**. O MLP 012 manteve melhor recall na classe 1 (0,7353; 25/34), enquanto a mistura teve melhor recall nas classes 6 e 13.

